<a href="https://colab.research.google.com/github/builtbyC/INFO570_term_project/blob/main/sbus_sa_test_1_4_3_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Cell 1 — Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# Edit this cell before every run. Everything else runs automatically.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Report Date ───────────────────────────────────────────────────────────────
# TEST MODE: date is hardcoded — change this value each run
REPORT_DATE_STR = "2026-02-23"
# PRODUCTION: uncomment the line below to auto-use today's date in EST
#REPORT_DATE_STR = __import__('datetime').datetime.now(__import__('zoneinfo').ZoneInfo("America/New_York")).strftime("%Y-%m-%d")

# ── Thresholds ────────────────────────────────────────────────────────────────
# Grade threshold: students scoring below this percentage are flagged as low-score
# e.g. 80 = 80%
GRADE_THRESHOLD = 80

# 'Action Needed' if missing assignments >= this number
MISSING_ALERT_THRESHOLD = 1

# 'Monitor' flag if late assignments >= this number
LATE_ALERT_THRESHOLD = 1

# 'Action Needed' if low-score assignments >= this number
LOW_SCORE_ALERT_THRESHOLD = 1

# ── Google Drive File IDs ─────────────────────────────────────────────────────
# How to get a file ID: right-click file in Google Drive → Share → Copy link
# The ID is the long string between /d/ and /view in the URL
SBUS_GRADES_FILE_ID     = "1G2r7G53mlEfbg8HE9egWdB6xpetugT4e"   # SBUS_Grades.csv
CANVAS_FILE_ID          = "18czzkDs3dkaom1K0YidIudbwcVSfN1hw"   # canvas_grades_detailed_YYYY-MM-DD.csv
SBUS_ENROLLMENT_FILE_ID = "11Ptft9Huo59KrlGVusz9f9-SIff8BVmJ"   # SBUS Enrollment and Tuition.xlsx

# ── Output Folder ─────────────────────────────────────────────────────────────
# Google Drive folder ID where the final report will be saved
# How to get folder ID: open the folder in Drive, copy the ID from the URL
OUTPUT_FOLDER_ID = "10JVSskfjLHkSISSONPl3NWsNjx5r9yoI" #need to change

# ═══════════════════════════════════════════════════════════════════════════════
# DO NOT edit below this line
# ═══════════════════════════════════════════════════════════════════════════════
print("Configuration loaded.")
print(f"   Report date     : {REPORT_DATE_STR}")
print(f"   Grade threshold : {GRADE_THRESHOLD}%")
print(f"   Missing alert   : >= {MISSING_ALERT_THRESHOLD}")
print(f"   Late alert      : >= {LATE_ALERT_THRESHOLD}")
print(f"   Low score alert : >= {LOW_SCORE_ALERT_THRESHOLD}")

Configuration loaded.
   Report date     : 2026-02-23
   Grade threshold : 80%
   Missing alert   : >= 1
   Late alert      : >= 1
   Low score alert : >= 1


In [8]:
# Cell 2 — Imports & File Loading
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import io
import warnings
from zoneinfo import ZoneInfo
from google.colab import drive, auth
from googleapiclient.discovery import build
import google.auth
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment

warnings.filterwarnings('ignore')

# ── Mount Google Drive ────────────────────────────────────────────────────────
# Access is restricted to minimum required scopes in compliance with FERPA:
#   - drive.readonly : read source files only
#   - drive.file     : write output report only (cannot access other Drive files)
drive.mount('/content/drive', force_remount=False)
auth.authenticate_user()

creds, _ = google.auth.default(scopes=[
    'https://www.googleapis.com/auth/drive.readonly',
    'https://www.googleapis.com/auth/drive.file'
])

drive_service = build('drive', 'v3', credentials=creds)

# ── Load Files from Google Drive by File ID ───────────────────────────────────
def load_drive_file(file_id, file_name):
    """
    Downloads a file from Google Drive by file ID and returns a DataFrame.
    Access is restricted to explicitly specified file IDs only.
    """
    print(f"   Loading {file_name}...")
    request = drive_service.files().get_media(fileId=file_id)
    buffer = io.BytesIO(request.execute())
    if file_name.lower().endswith('.csv'):
        return pd.read_csv(buffer)
    elif file_name.lower().endswith('.xlsx'):
        return pd.read_excel(buffer)
    else:
        raise ValueError(f"Unsupported file format: {file_name}")

# ── Load Source Files ─────────────────────────────────────────────────────────
print("Loading source files from Google Drive...\n")

sbus_grades_df     = load_drive_file(SBUS_GRADES_FILE_ID,     "SBUS_Grades.csv")
canvas_df          = load_drive_file(CANVAS_FILE_ID,          "canvas_grades_detailed.csv")
sbus_enrollment_df = load_drive_file(SBUS_ENROLLMENT_FILE_ID, "SBUS Enrollment and Tuition.xlsx")

# ── Date & Output Configuration ───────────────────────────────────────────────
EST             = ZoneInfo("America/New_York")
report_date_est = pd.Timestamp(REPORT_DATE_STR, tz=EST)
report_date     = report_date_est.astimezone(ZoneInfo("UTC"))
grade_threshold = GRADE_THRESHOLD / 100  # convert percentage to fraction for internal calculations

missing_alert_threshold   = MISSING_ALERT_THRESHOLD
late_alert_threshold      = LATE_ALERT_THRESHOLD
low_score_alert_threshold = LOW_SCORE_ALERT_THRESHOLD

OUTPUT_FILE = f"SBUS_Advisor_Report_{report_date_est.strftime('%Y%m%d')}.xlsx"

# ── Confirmation ──────────────────────────────────────────────────────────────
print("\n All files loaded successfully.")
print(f"   SBUS Grades        : {sbus_grades_df.shape[0]} rows, {sbus_grades_df.shape[1]} columns")
print(f"   Canvas Report      : {canvas_df.shape[0]} rows, {canvas_df.shape[1]} columns")
print(f"   SBUS Enrollment    : {sbus_enrollment_df.shape[0]} rows, {sbus_enrollment_df.shape[1]} columns")
print(f"\n   Report Date        : {report_date_est.date()} EST")
print(f"   Grade Threshold    : {GRADE_THRESHOLD}%")
print(f"   Output File        : {OUTPUT_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading source files from Google Drive...

   Loading SBUS_Grades.csv...
   Loading canvas_grades_detailed.csv...
   Loading SBUS Enrollment and Tuition.xlsx...

 All files loaded successfully.
   SBUS Grades        : 21289 rows, 23 columns
   Canvas Report      : 3345 rows, 22 columns
   SBUS Enrollment    : 5109 rows, 51 columns

   Report Date        : 2026-02-23 EST
   Grade Threshold    : 80%
   Output File        : SBUS_Advisor_Report_20260223.xlsx


In [12]:
# Cell 3 — Processing & Output
# ═══════════════════════════════════════════════════════════════════════════════

# ── Column Order Configuration ────────────────────────────────────────────────
# To reorder: move an item to a different group, or move a whole group up/down.
# Assignment columns are always appended last, auto-sorted by due date.
STATIC_COL_GROUPS = [
    ["NAME", "ID", "MSU_EMAIL", "PHONE_NUMBER"],
    ["REGISTRATION_COHORT", "CUMULATIVE_GPA", "CURRENT_COURSE_IN_PROGRESS"],
    ["current_score"],
    ["total_assignments", "total_missing", "total_late", "total_low_score"],
    # Assignment columns follow automatically — do not add them here
]

# ── Enrollment Columns to Carry Into Each Sheet ───────────────────────────────
ENROLLMENT_KEEP = [
    'ID', 'NAME', 'MSU_EMAIL', 'PHONE_NUMBER',
    'REGISTRATION_COHORT', 'CUMULATIVE_GPA',
    'CURRENT_COURSE_IN_PROGRESS',
]

# ── Static Column Display Name Mapping ───────────────────────────────────────
# Maps internal column names to display names shown in the Excel report.
# Enrollment columns are already uppercase — only lowercase internal cols need mapping.
DISPLAY_NAMES = {
    'current_score'     : 'Current Score',
    'total_assignments' : 'Total Assignments',
    'total_missing'     : 'Total Missing',
    'total_late'        : 'Total Late',
    'total_low_score'   : 'Total Low Score',
}

# ── Cell Fill Colors ──────────────────────────────────────────────────────────
COLOR_MISSING   = "FF0000"   # Red          — action needed
COLOR_LATE      = "FFC000"   # Amber        — monitor
COLOR_LOW_SCORE = "FFFF99"   # Light Yellow — caution
COLOR_UNKNOWN   = "C0C0C0"   # Light Grey   — no Canvas row found, investigate

# ── Preprocess: Canvas (Source of Truth) ─────────────────────────────────────
df_canvas = canvas_df.copy()
df_canvas.columns = df_canvas.columns.str.strip().str.lower()
df_canvas['sis_user_id']             = df_canvas['sis_user_id'].astype(str).str.strip()
df_canvas['assignment_name']         = df_canvas['assignment_name'].str.strip()
df_canvas['assignment_due_at']       = pd.to_datetime(df_canvas['assignment_due_at'],       utc=True, errors='coerce')
df_canvas['assignment_submitted_at'] = pd.to_datetime(df_canvas['assignment_submitted_at'], utc=True, errors='coerce')

# Drop rows where due date could not be parsed
df_canvas = df_canvas[df_canvas['assignment_due_at'].notna()].copy()

# Include only assignments due on or before the report date
df_canvas = df_canvas[df_canvas['assignment_due_at'] <= report_date].copy()

# Drop assignments with a max possible score of 0 — placeholder or ungraded items
df_canvas = df_canvas[df_canvas['points_possible'] > 0].copy()

# Capture dropped assignments per course for logging
before_filter = df_canvas[['course_name', 'assignment_name']].drop_duplicates()

# Drop the FSBUS Academic Integrity Pledge — administrative item, not a graded assignment
df_canvas = df_canvas[~df_canvas['assignment_name'].str.contains(
    'FSBUS Academic Integrity Pledge', case=False, na=False)].copy()

after_filter = df_canvas[['course_name', 'assignment_name']].drop_duplicates()

dropped_by_course = (
    before_filter[~before_filter.set_index(['course_name', 'assignment_name']).index
    .isin(after_filter.set_index(['course_name', 'assignment_name']).index)]
    .groupby('course_name')['assignment_name']
    .apply(list)
    .to_dict()
)

# Normalize late and missing flags to boolean
for col in ['late', 'missing']:
    df_canvas[col] = df_canvas[col].astype(str).str.upper().isin(['TRUE', '1', 'YES'])

# Normalize status column to lowercase for consistent comparison
df_canvas['status'] = df_canvas['status'].astype(str).str.strip().str.lower()

# ── Preprocess: SBUS Grades (Current Score Only) ──────────────────────────────
df_grades = sbus_grades_df.copy()
df_grades.columns = df_grades.columns.str.strip().str.lower().str.replace(' ', '_')
df_grades = df_grades[df_grades['enrollment_state'] == 'active'].copy()
df_grades['student_sis'] = df_grades['student_sis'].astype(str).str.strip()

# ── Preprocess: Enrollment (Demographics Only) ────────────────────────────────
df_enrollment = sbus_enrollment_df.copy()

# Strip leading M from student ID to align with Canvas and Grades identifiers
df_enrollment['ID'] = df_enrollment['ID'].astype(str).str.strip().str.lstrip('M')
df_enrollment_slim = df_enrollment[[c for c in ENROLLMENT_KEEP if c in df_enrollment.columns]].copy()

print(f"Preprocessing complete.")
print(f"   Canvas rows        : {len(df_canvas)}")
print(f"   SBUS Grades rows   : {len(df_grades)}")
print(f"   Enrollment rows    : {len(df_enrollment_slim)}")

# ── Helper Functions ──────────────────────────────────────────────────────────
def get_assignment_status(arow):
    """
    Determines assignment status from a Canvas row.

    Rules:
      - Missing   : missing = TRUE and status = 'no submission'
      - Late      : late = TRUE and submitted_at > due_at
      - Submitted : status = 'submitted' and submitted_at exists

    Missing takes priority — a student cannot be late if they never submitted.
    """
    is_missing = (
        arow['missing'] == True and
        arow['status'] == 'no submission'
    )
    is_late = (
        arow['late'] == True and
        pd.notna(arow['assignment_submitted_at']) and
        pd.notna(arow['assignment_due_at']) and
        arow['assignment_submitted_at'] > arow['assignment_due_at']
    )

    if is_missing:
        return 'missing'
    if is_late:
        return 'late'
    return 'submitted'


def format_assignment_header(assignment_name, suffix):
    """
    Formats an assignment column header by wrapping the assignment name
    at 25 characters per line, then appending Status or Grade on its own
    final line. No text is truncated.
    """
    words = assignment_name.split(' ')
    lines = []
    current_line = ''
    for word in words:
        if len(current_line) + len(word) + (1 if current_line else 0) <= 25:
            current_line = f"{current_line} {word}".strip()
        else:
            if current_line:
                lines.append(current_line)
            while len(word) > 25:
                lines.append(word[:25])
                word = word[25:]
            current_line = word
    if current_line:
        lines.append(current_line)

    label = 'Status' if suffix == 'status' else 'Grade'
    lines.append(label)

    return '\n'.join(lines)


def build_course_df(df_cv, df_grades_course):
    """
    Builds a wide-format DataFrame for a single course.

    Source of truth : Canvas — determines students and assignments
    SBUS Grades     : provides current_score only
    Enrollment      : provides demographics only (merged after)

    Status definitions:
      - missing   : missing = TRUE and status = 'no submission'
      - late      : late = TRUE and submitted_at > due_at
      - unknown   : no Canvas row found for this student/assignment
      - submitted : status = 'submitted' and submitted_at exists

    Low score: applied to any graded assignment (submitted or late)
    scoring below the grade threshold. Missing and unknown excluded.
    """
    # Get unique students from Canvas for this course
    student_ids = df_cv['sis_user_id'].unique()

    # Get assignments sorted by due date
    assignments = (
        df_cv[['assignment_name', 'assignment_due_at']]
        .drop_duplicates('assignment_name')
        .sort_values('assignment_due_at')['assignment_name']
        .tolist()
    )

    records = []
    for sid in student_ids:
        sc = df_cv[df_cv['sis_user_id'] == sid]

        total_missing   = 0
        total_late      = 0
        total_low_score = 0
        record = {'student_sis': sid}

        for asgn in assignments:
            arow = sc[sc['assignment_name'] == asgn]

            if arow.empty:
                # No Canvas row found for this student/assignment — flag as unknown
                status = 'unknown'
                grade  = np.nan
            else:
                arow   = arow.iloc[0]
                status = get_assignment_status(arow)

                # Calculate grade for graded assignments only
                if (
                    status != 'missing' and
                    pd.notna(arow['score']) and
                    pd.notna(arow['points_possible']) and
                    arow['points_possible'] > 0
                ):
                    grade = round(arow['score'] / arow['points_possible'] * 100, 2)
                else:
                    grade = np.nan

            # Aggregate counts
            if status == 'missing':
                total_missing += 1
            elif status == 'late':
                total_late += 1

            # Low score: graded assignments only, missing and unknown excluded
            if not np.isnan(grade) and (grade / 100) < grade_threshold:
                total_low_score += 1

            record[f'{asgn}_status'] = status
            record[f'{asgn}_grade']  = grade

        record['total_assignments'] = len(assignments)
        record['total_missing']     = total_missing
        record['total_late']        = total_late
        record['total_low_score']   = total_low_score

        records.append(record)

    df_wide = pd.DataFrame(records)

    # Merge current_score from SBUS Grades
    if not df_grades_course.empty and 'current_score' in df_grades_course.columns:
        score_lookup = df_grades_course[['student_sis', 'current_score']].drop_duplicates('student_sis')
        df_wide = df_wide.merge(score_lookup, on='student_sis', how='left')
    else:
        df_wide['current_score'] = np.nan

    return df_wide, assignments


def apply_col_order(df, assignment_cols):
    """Order columns per STATIC_COL_GROUPS; assignment columns always last."""
    static = [c for group in STATIC_COL_GROUPS for c in group if c in df.columns]
    return df[[c for c in static + assignment_cols if c in df.columns]]


def safe_sheet_name(name, used, max_len=31):
    """Truncate to Excel 31-character sheet name limit and ensure uniqueness."""
    base = name[:max_len]
    if base not in used:
        return base
    for i in range(1, 100):
        candidate = f"{name[:max_len - 3]}_{i}"
        if candidate not in used:
            return candidate
    raise ValueError(f"Could not generate a unique sheet name for: {name}")


def apply_excel_formatting(ws, assignments):
    """
    Applies formatting to a worksheet:
      - Header row: dark blue background, white bold text, wrap text
      - Column width fixed at 25 characters
      - Assignment headers wrapped at 25 chars with Status/Grade on final line
      - Static column headers mapped to Title Case display names
      - Data rows: color coded for missing, late, low score, and unknown
    """
    from openpyxl.styles import Alignment, Font, PatternFill

    HEADER_FILL    = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    HEADER_FONT    = Font(color="FFFFFF", bold=True)
    FILL_MISSING   = PatternFill(start_color=COLOR_MISSING,   end_color=COLOR_MISSING,   fill_type="solid")
    FILL_LATE      = PatternFill(start_color=COLOR_LATE,      end_color=COLOR_LATE,      fill_type="solid")
    FILL_LOW_SCORE = PatternFill(start_color=COLOR_LOW_SCORE, end_color=COLOR_LOW_SCORE, fill_type="solid")
    FILL_UNKNOWN   = PatternFill(start_color=COLOR_UNKNOWN,   end_color=COLOR_UNKNOWN,   fill_type="solid")

    # Build lookup of internal column name -> formatted assignment header
    formatted_headers = {}
    for asgn in assignments:
        formatted_headers[f'{asgn}_status'] = format_assignment_header(asgn, 'status')
        formatted_headers[f'{asgn}_grade']  = format_assignment_header(asgn, 'grade')

    # Build lookup of column index -> internal column name for data row coloring
    col_index_to_internal = {}
    max_lines = 1

    # ── Format Header Row ─────────────────────────────────────────────────────
    for cell in ws[1]:
        col_letter = cell.column_letter
        internal   = cell.value

        col_index_to_internal[cell.column] = internal

        if internal in DISPLAY_NAMES:
            cell.value = DISPLAY_NAMES[internal]
        elif internal in formatted_headers:
            cell.value = formatted_headers[internal]

        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = Alignment(wrap_text=True, vertical='top', horizontal='center')

        if cell.value and isinstance(cell.value, str):
            max_lines = max(max_lines, cell.value.count('\n') + 1)

        ws.column_dimensions[col_letter].width = 25

    # Set header row height based on number of wrapped lines
    ws.row_dimensions[1].height = max_lines * 15

    # ── Color Code Data Rows ──────────────────────────────────────────────────
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            internal = col_index_to_internal.get(cell.column, '')

            # Color assignment status columns
            if internal and internal.endswith('_status'):
                if cell.value == 'missing':
                    cell.fill = FILL_MISSING
                elif cell.value == 'late':
                    cell.fill = FILL_LATE
                elif cell.value == 'unknown':
                    cell.fill = FILL_UNKNOWN

            # Color assignment grade columns if below threshold
            if internal and internal.endswith('_grade'):
                if cell.value is not None and not isinstance(cell.value, str):
                    if cell.value < GRADE_THRESHOLD:
                        cell.fill = FILL_LOW_SCORE

            # Color summary count columns
            if internal == 'total_missing' and cell.value and cell.value >= missing_alert_threshold:
                cell.fill = FILL_MISSING
            if internal == 'total_late' and cell.value and cell.value >= late_alert_threshold:
                cell.fill = FILL_LATE
            if internal == 'total_low_score' and cell.value and cell.value >= low_score_alert_threshold:
                cell.fill = FILL_LOW_SCORE


# ── Build One DataFrame Per Course ────────────────────────────────────────────
course_sheets = {}

for course_code in df_canvas['course_name'].unique():
    df_cv = df_canvas[df_canvas['course_name'] == course_code].copy()

    # Pull current_score from SBUS Grades for students in this course
    df_grades_course = df_grades[df_grades['course'] == course_code].copy() \
        if 'course' in df_grades.columns else pd.DataFrame()

    descriptive_name = df_cv['course_section'].iloc[0]
    df_wide, assignments = build_course_df(df_cv, df_grades_course)

    asgn_cols = [
        col
        for asgn in assignments
        for col in (f'{asgn}_status', f'{asgn}_grade')
    ]

    # Merge demographics from enrollment file
    df_wide = df_wide.merge(df_enrollment_slim, left_on='student_sis', right_on='ID', how='left')
    course_sheets[descriptive_name] = (apply_col_order(df_wide, asgn_cols), assignments)

    print(f"   {descriptive_name}: {len(df_wide)} students, {len(assignments)} assignments")
    dropped = dropped_by_course.get(course_code, [])
    if dropped:
        print(f"   Dropped assignments in {descriptive_name}:")
        for name in dropped:
            print(f"      - {name}")

# ── Build Summary Sheet ───────────────────────────────────────────────────────
# One row per student; per-course: current_score, total_assignments,
# total_missing, total_late, total_low_score
summary_parts = []

for course_name, (df_course, _) in course_sheets.items():
    id_col = 'ID' if 'ID' in df_course.columns else 'student_sis'
    keep   = [c for c in [
                  id_col, 'NAME',
                  'current_score',
                  'total_assignments',
                  'total_missing', 'total_late', 'total_low_score'
              ] if c in df_course.columns]
    temp   = df_course[keep].copy()

    prefix = course_name[:20].strip()
    temp   = temp.rename(columns={
        'current_score'     : f'{prefix}_current_score',
        'total_assignments' : f'{prefix}_total_assignments',
        'total_missing'     : f'{prefix}_missing',
        'total_late'        : f'{prefix}_late',
        'total_low_score'   : f'{prefix}_low_score',
    })
    summary_parts.append(temp)

df_summary_out = summary_parts[0]
id_col = 'ID' if 'ID' in df_summary_out.columns else 'student_sis'

for part in summary_parts[1:]:
    df_summary_out = df_summary_out.merge(part, on=[id_col, 'NAME'], how='outer')

print(f"\nSummary sheet: {df_summary_out.shape[0]} students, {df_summary_out.shape[1]} columns")

# ── Write Excel & Apply Formatting ───────────────────────────────────────────
print(f"\nWriting report...")

# Build a clean sheet name mapping once to ensure consistency
sheet_name_map = {}
used_names = {'Summary'}
for course_name in course_sheets:
    sheet_name_map[course_name] = safe_sheet_name(course_name, used_names)
    used_names.add(sheet_name_map[course_name])

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_summary_out.to_excel(writer, sheet_name='Summary', index=False)

    for course_name, (df_course, assignments) in course_sheets.items():
        sheet_name = sheet_name_map[course_name]
        df_course.to_excel(writer, sheet_name=sheet_name, index=False)

    # Apply formatting to Summary sheet
    apply_excel_formatting(writer.sheets['Summary'], [])

    # Apply formatting to each course sheet
    for course_name, (df_course, assignments) in course_sheets.items():
        sheet_name = sheet_name_map[course_name]
        apply_excel_formatting(writer.sheets[sheet_name], assignments)

# ── Upload Report to Google Drive ─────────────────────────────────────────────
from googleapiclient.http import MediaFileUpload

file_metadata = {
    'name'    : OUTPUT_FILE,
    'parents' : [OUTPUT_FOLDER_ID]
}

media = MediaFileUpload(
    OUTPUT_FILE,
    mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)

uploaded_file = drive_service.files().create(
    body=file_metadata,
    media_body=media,
    fields='id, name, webViewLink'
).execute()

print(f"\nReport saved successfully.")
print(f"   File name  : {uploaded_file.get('name')}")
print(f"   Drive link : {uploaded_file.get('webViewLink')}")

Preprocessing complete.
   Canvas rows        : 995
   SBUS Grades rows   : 20939
   Enrollment rows    : 5109
   MKTG562_01SP26: 4 students, 6 assignments
   INFO584_91SP26: 4 students, 25 assignments
   Dropped assignments in INFO584_91SP26:
      - FSBUS Academic Integrity Pledge (Required to Access Modules)
   INFO584_92SP26: 14 students, 25 assignments
   Dropped assignments in INFO584_92SP26:
      - FSBUS Academic Integrity Pledge (Required to Access Modules)
   MGMT562_70SP26: 2 students, 33 assignments
   INFO591_91SP26: 5 students, 1 assignments
   INFO588_91SP26: 45 students, 5 assignments

Summary sheet: 68 students, 32 columns

Writing report...

Report saved successfully.
   File name  : SBUS_Advisor_Report_20260223.xlsx
   Drive link : https://docs.google.com/spreadsheets/d/1p1DkCCb-hEuxhsCZcgK662rkl93tO6Mr/edit?usp=drivesdk&ouid=117345034838892549489&rtpof=true&sd=true
